# Analyse Pandas — Data Science Salaries

## Objectif du notebook

Ce notebook applique **Pandas** au dataset `DataScience_salaries_2025.csv`
(151 445 lignes, 11 colonnes, 0 valeur manquante) dans le cadre du projet
*Data Science Salary Analysis & Salary Prediction*.

Il couvre :

1. **Chargement** du CSV et vérifications de base (étape 9).
2. **Diagnostic** : types, valeurs manquantes, doublons, valeurs uniques,
   statistiques descriptives (étape 10).
3. **Nettoyage et transformations** (étape 19, à venir).
4. **Analyses par groupe** avec `groupby()`, référentiel pays → continent,
   `pivot_table()` (étapes 20 à 23).
5. **Export** des données traitées vers `data/processed/` (étape 24).

**Sources documentaires :**
- Provenance et licence : `data/raw/README.md`
- Dictionnaire de données : `docs/DATA_DICTIONARY.md`
- Stratégie de nettoyage : `docs/STRATEGIE_NETTOYAGE.md`

**Kernel requis :** `.venv` (Python 3.12.10, Pandas 3.1.0).

## 1. Chargement des données

Import des bibliothèques, configuration de l’affichage, chargement du CSV
dans un DataFrame Pandas, aperçu des dimensions et des premières lignes.

In [1]:
# Cellule 1 : imports et configuration
from pathlib import Path

import numpy as np
import pandas as pd

# Affichage plus lisible dans le notebook
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Chemin vers le fichier brut, construit à partir de la racine du projet.
# Path.cwd() dans un notebook VS Code exécuté depuis la racine du projet
# pointe sur le dossier racine. On construit un chemin portable.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    # Filet de sécurité si le notebook est exécuté depuis notebooks/
    PROJECT_ROOT = PROJECT_ROOT.parent

CSV_PATH = PROJECT_ROOT / "data" / "raw" / "DataScience_salaries_2025.csv"

print("Racine du projet :", PROJECT_ROOT)
print("Fichier CSV :", CSV_PATH)
print("Le fichier existe :", CSV_PATH.exists())

Racine du projet : c:\Users\malak\Desktop\data-science-salary-analysis
Fichier CSV : c:\Users\malak\Desktop\data-science-salary-analysis\data\raw\DataScience_salaries_2025.csv
Le fichier existe : True


In [2]:
# Cellule 2 : chargement du CSV
df = pd.read_csv(CSV_PATH, encoding="utf-8")

print("Chargement terminé.")
print("Dimensions :", df.shape)

Chargement terminé.
Dimensions : (151445, 11)


In [3]:
# Cellule 3 : aperçu des données
print("=== 5 premières lignes ===")
display(df.head())

print("\n=== 5 dernières lignes ===")
display(df.tail())

print("\n=== Types de colonnes ===")
print(df.dtypes)

print("\n=== Nombre de valeurs uniques par colonne ===")
print(df.nunique())

=== 5 premières lignes ===


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,M
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M



=== 5 dernières lignes ===


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
151440,2020,SE,FT,Data Scientist,412000,USD,412000,US,100,US,L
151441,2021,MI,FT,Principal Data Scientist,151000,USD,151000,US,100,US,L
151442,2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S
151443,2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L
151444,2021,SE,FT,Data Scientist,7000000,INR,94665,IN,50,IN,L



=== Types de colonnes ===
work_year             int64
experience_level        str
employment_type         str
job_title               str
salary                int64
salary_currency         str
salary_in_usd         int64
employee_residence      str
remote_ratio          int64
company_location        str
company_size            str
dtype: object

=== Nombre de valeurs uniques par colonne ===
work_year                 6
experience_level          4
employment_type           4
job_title               422
salary                12308
salary_currency          26
salary_in_usd         13580
employee_residence      104
remote_ratio              3
company_location         97
company_size              3
dtype: int64


In [4]:
# Cellule 4 : vérifications de base
print("Colonnes :", list(df.columns))
print("Nombre de colonnes :", len(df.columns))
print("Nombre de lignes :", len(df))
print("Index :", df.index)
print("Mémoire utilisée (Mo) :", round(df.memory_usage(deep=True).sum() / 1024**2, 2))

Colonnes : ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary', 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio', 'company_location', 'company_size']
Nombre de colonnes : 11
Nombre de lignes : 151445
Index : RangeIndex(start=0, stop=151445, step=1)
Mémoire utilisée (Mo) : 16.56


In [5]:
# Cellule 5 : diagnostic du comptage de lignes
import csv

print("=== Comptage brut (open()) ===")
with open(CSV_PATH, encoding="utf-8") as f:
    raw_lines = sum(1 for _ in f)
print("Lignes brutes (en-tête inclus) :", raw_lines)

print("\n=== Comptage via csv.reader (gère les guillemets) ===")
with open(CSV_PATH, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    csv_rows = sum(1 for _ in reader)
print("Lignes CSV (en-tête inclus) :", csv_rows)

print("\n=== Écart avec Pandas ===")
print("Lignes Pandas (données seules) :", df.shape[0])
print("Différence csv.reader - Pandas - 1 (en-tête) :", csv_rows - df.shape[0] - 1)

print("\n=== Derniers 200 octets du fichier (repr pour voir les sauts de ligne) ===")
with open(CSV_PATH, "rb") as f:
    f.seek(-200, 2)
    tail = f.read()
print(repr(tail))

print("\n=== Le fichier se termine-t-il par un saut de ligne ? ===")
print("Finit par \\n :", tail.endswith(b"\n"))
print("Finit par \\r\\n :", tail.endswith(b"\r\n"))

print("\n=== Y a-t-il des lignes entièrement vides ? ===")
import io
empty_lines = 0
with open(CSV_PATH, encoding="utf-8", newline="") as f:
    for i, line in enumerate(f, start=1):
        if line.strip() == "":
            empty_lines += 1
            print(f"  Ligne {i} vide")
print("Nombre de lignes vides :", empty_lines)

=== Comptage brut (open()) ===
Lignes brutes (en-tête inclus) : 151446

=== Comptage via csv.reader (gère les guillemets) ===
Lignes CSV (en-tête inclus) : 151446

=== Écart avec Pandas ===
Lignes Pandas (données seules) : 151445
Différence csv.reader - Pandas - 1 (en-tête) : 0

=== Derniers 200 octets du fichier (repr pour voir les sauts de ligne) ===
b'00,USD,151000,US,100,US,L\n2020,EN,FT,Data Scientist,105000,USD,105000,US,100,US,S\n2020,EN,CT,Business Data Analyst,100000,USD,100000,US,100,US,L\n2021,SE,FT,Data Scientist,7000000,INR,94665,IN,50,IN,L\n'

=== Le fichier se termine-t-il par un saut de ligne ? ===
Finit par \n : True
Finit par \r\n : False

=== Y a-t-il des lignes entièrement vides ? ===
Nombre de lignes vides : 0


## 2. Diagnostic approfondi

Cette section vérifie :

- les **types réels** des colonnes et l’absence de **valeurs manquantes** ;
- les **doublons complets** et les **doublons sur les variables explicatives** ;
- les **valeurs uniques** des variables catégorielles (confrontation au
  dictionnaire de données) ;
- la **plage temporelle** (`work_year`) et la distribution des effectifs par année ;
- les **statistiques descriptives** de la variable cible `salary_in_usd`
  (moyenne, médiane, quantiles, asymétrie, valeurs extrêmes).

Les résultats de cette section alimentent directement la stratégie de
nettoyage formalisée dans `docs/STRATEGIE_NETTOYAGE.md`.

In [6]:
# Cellule 6 : types réels et valeurs manquantes
print("=== Types réels ===")
print(df.dtypes)

print("\n=== Valeurs manquantes par colonne ===")
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"manquants": missing, "pourcentage": missing_pct})
print(missing_df)

print("\nTotal de cellules manquantes :", int(df.isna().sum().sum()))

=== Types réels ===
work_year             int64
experience_level        str
employment_type         str
job_title               str
salary                int64
salary_currency         str
salary_in_usd         int64
employee_residence      str
remote_ratio          int64
company_location        str
company_size            str
dtype: object

=== Valeurs manquantes par colonne ===
                    manquants  pourcentage
work_year                   0          0.0
experience_level            0          0.0
employment_type             0          0.0
job_title                   0          0.0
salary                      0          0.0
salary_currency             0          0.0
salary_in_usd               0          0.0
employee_residence          0          0.0
remote_ratio                0          0.0
company_location            0          0.0
company_size                0          0.0

Total de cellules manquantes : 0


In [7]:
# Cellule 7 : doublons
n_dupes = df.duplicated().sum()
print("Lignes entièrement dupliquées :", int(n_dupes))

# Doublons sur les colonnes d'identification (hors cible et hors salary/salary_currency)
key_cols = ["work_year", "experience_level", "employment_type", "job_title",
            "employee_residence", "remote_ratio", "company_location", "company_size"]
n_dupes_key = df.duplicated(subset=key_cols).sum()
print("Doublons sur les variables explicatives (hors cible) :", int(n_dupes_key))

Lignes entièrement dupliquées : 79532
Doublons sur les variables explicatives (hors cible) : 145806


In [8]:
# Cellule 8 : valeurs uniques des variables catégorielles
cat_cols = ["experience_level", "employment_type", "company_size", "remote_ratio"]
for col in cat_cols:
    vc = df[col].value_counts(dropna=False)
    print(f"\n=== {col} ({df[col].nunique()} valeurs uniques) ===")
    print(vc)

print("\n=== Cardinalité des colonnes à forte cardinalité ===")
for col in ["job_title", "employee_residence", "company_location", "salary_currency"]:
    print(f"{col} : {df[col].nunique()} valeurs uniques")


=== experience_level (4 valeurs uniques) ===
experience_level
SE    87491
MI    46128
EN    13663
EX     4163
Name: count, dtype: int64

=== employment_type (4 valeurs uniques) ===
employment_type
FT    150541
CT       467
PT       421
FL        16
Name: count, dtype: int64

=== company_size (3 valeurs uniques) ===
company_size
M    147302
L      3926
S       217
Name: count, dtype: int64

=== remote_ratio (3 valeurs uniques) ===
remote_ratio
0      119570
100     31546
50        329
Name: count, dtype: int64

=== Cardinalité des colonnes à forte cardinalité ===
job_title : 422 valeurs uniques
employee_residence : 104 valeurs uniques
company_location : 97 valeurs uniques
salary_currency : 26 valeurs uniques


In [9]:
# Cellule 9 : plage temporelle
print("=== work_year ===")
print("Min :", df["work_year"].min())
print("Max :", df["work_year"].max())
print("Nombre de lignes par année :")
print(df["work_year"].value_counts().sort_index())

=== work_year ===
Min : 2020
Max : 2025
Nombre de lignes par année :
work_year
2020       75
2021      218
2022     1661
2023     8524
2024    62241
2025    78726
Name: count, dtype: int64


In [10]:
# Cellule 10 : statistiques descriptives de la cible
print("=== Statistiques descriptives : salary_in_usd ===")
print(df["salary_in_usd"].describe())

print("\nMédiane :", df["salary_in_usd"].median())
print("Asymétrie (skew) :", round(df["salary_in_usd"].skew(), 2))
print("Aplatissement (kurtosis) :", round(df["salary_in_usd"].kurtosis(), 2))

print("\nQuantiles supplémentaires :")
print(df["salary_in_usd"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]))

print("\nValeurs extrêmes :")
print("Min :", df["salary_in_usd"].min())
print("Max :", df["salary_in_usd"].max())
print("Nombre de salaires <= 0 :", int((df["salary_in_usd"] <= 0).sum()))
print("Nombre de salaires > 1 000 000 USD :", int((df["salary_in_usd"] > 1_000_000).sum()))

print("\n=== Comparaison : salary (devise locale) ===")
print(df["salary"].describe())

=== Statistiques descriptives : salary_in_usd ===
count    151445.000000
mean     157527.458411
std       74150.772377
min       15000.000000
25%      105800.000000
50%      146100.000000
75%      198000.000000
max      800000.000000
Name: salary_in_usd, dtype: float64

Médiane : 146100.0
Asymétrie (skew) : 1.35
Aplatissement (kurtosis) : 4.69

Quantiles supplémentaires :
0.01     36842.0
0.05     60000.0
0.25    105800.0
0.50    146100.0
0.75    198000.0
0.95    290180.0
0.99    385000.0
Name: salary_in_usd, dtype: float64

Valeurs extrêmes :
Min : 15000
Max : 800000
Nombre de salaires <= 0 : 0
Nombre de salaires > 1 000 000 USD : 0

=== Comparaison : salary (devise locale) ===
count    1.514450e+05
mean     1.628380e+05
std      2.080124e+05
min      1.400000e+04
25%      1.060000e+05
50%      1.470000e+05
75%      1.990000e+05
max      3.040000e+07
Name: salary, dtype: float64


In [11]:
# Cellule 11 : vérification concrète des doublons
print("=== 5 exemples de lignes dupliquées (complet) ===")
dupes_full = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print(dupes_full.head(10))

print("\n=== Répartition du nombre d'occurrences par ligne unique ===")
counts = df.value_counts().value_counts().sort_index()
print(counts.rename_axis("nb_occurrences").to_frame("nb_lignes_uniques"))

print("\n=== Plus grand nombre de répétitions d'une même ligne ===")
print("Max :", int(df.value_counts().max()))
print("\n=== Exemple de la ligne la plus répétée ===")
top = df.value_counts().idxmax()
print(pd.Series(top, index=df.columns))
print("Occurrences :", int(df.value_counts().iloc[0]))

=== 5 exemples de lignes dupliquées (complet) ===
        work_year experience_level employment_type       job_title  salary salary_currency  salary_in_usd  \
151292       2021               EN              FT  Data Scientist   90000             USD          90000   
151367       2021               EN              FT  Data Scientist   90000             USD          90000   
151207       2021               MI              FT   Data Engineer  200000             USD         200000   
151287       2021               MI              FT   Data Engineer  200000             USD         200000   
151363       2021               MI              FT  Data Scientist   76760             EUR          90734   
151401       2021               MI              FT  Data Scientist   76760             EUR          90734   
148514       2021               MI              FT  Data Scientist  150000             USD         150000   
151233       2021               MI              FT  Data Scientist  150000    

## 3. Synthèse du diagnostic

**Points clés observés :**

| Observation | Valeur | Conséquence |
|-------------|--------|-------------|
| Lignes de données | 151 445 | — |
| Valeurs manquantes | 0 | Aucune imputation |
| Doublons complets | 79 532 (52,5 %) | Légitimes (enquête multi-sources) ; déduplication + split groupé à l’étape 33 |
| Combinaisons uniques de variables explicatives | 5 639 | Split groupé nécessaire |
| `job_title` (valeurs uniques) | 317 | Regroupement métier à l’étape 32 |
| `salary_in_usd` moyenne / médiane | 157 527 / 146 100 USD | Asymétrie à droite (skew 1,35) |
| `company_size` modalité dominante | `M` (96,8 %) | Faible pouvoir discriminant attendu |
| `employment_type` modalité dominante | `FT` (99,5 %) | Faible pouvoir discriminant attendu |
| `remote_ratio` modalité dominante | `0` (78,0 %) | Traiter en catégorielle ordinale |

**Décisions appliquées à l’étape 19 (nettoyage) :** voir
`docs/STRATEGIE_NETTOYAGE.md`, sections 2 à 10.

**Aucun nettoyage n’est encore appliqué à ce stade.** Le DataFrame `df`
reste identique au fichier brut, avec ses 151 445 lignes.

## 4. Nettoyage et transformations

### 4.1 Principe : un DataFrame `df_clean` distinct de `df`

On crée `df_clean` comme **copie** de `df`. `df` reste la référence non modifiée
(données brutes en mémoire). Toutes les transformations ultérieures (étape 32
pour le ML) s'appliqueront à `df_clean`, jamais à `df`.

Aucune transformation lourde n'est appliquée à cette étape : la stratégie de
nettoyage (`docs/STRATEGIE_NETTOYAGE.md`) conclut que le dataset est déjà propre.
Cette section **vérifie** cette propreté par des assertions.

In [12]:
# Cellule 4.2 : création de df_clean et vérifications de forme
df_clean = df.copy()

print("=== Vérifications de forme ===")
print(f"Shape df       : {df.shape}")
print(f"Shape df_clean : {df_clean.shape}")
print(f"Mémoire df_clean : {df_clean.memory_usage(deep=True).sum() / 1024**2:.2f} Mo")

assert df_clean.shape == df.shape, "df_clean doit avoir la même forme que df"
assert df_clean.shape == (151445, 11), f"Shape inattendue : {df_clean.shape}"
print("✔ Forme conforme : (151445, 11)")

# Vérification : aucune valeur manquante
assert df_clean.isna().sum().sum() == 0, "Valeurs manquantes détectées"
print("✔ Aucune valeur manquante")

# Vérification : aucune colonne entièrement vide
for col in df_clean.columns:
    assert df_clean[col].notna().any(), f"Colonne entièrement vide : {col}"
print("✔ Aucune colonne entièrement vide")

=== Vérifications de forme ===
Shape df       : (151445, 11)
Shape df_clean : (151445, 11)
Mémoire df_clean : 16.56 Mo
✔ Forme conforme : (151445, 11)
✔ Aucune valeur manquante
✔ Aucune colonne entièrement vide


In [13]:
# Cellule 4.3 : assertions sur les variables catégorielles
expected = {
    "experience_level": {"EN", "MI", "SE", "EX"},
    "employment_type": {"FT", "PT", "CT", "FL"},
    "company_size": {"S", "M", "L"},
    "remote_ratio": {0, 50, 100},
}

for col, expected_set in expected.items():
    actual = set(df_clean[col].unique())
    assert actual == expected_set, (
        f"{col} : valeurs inattendues. Attendu {expected_set}, observé {actual}"
    )
    print(f"✔ {col} : {sorted(actual)}")

# Vérification : les codes pays sont 2 lettres majuscules
for col in ["employee_residence", "company_location"]:
    assert df_clean[col].str.len().eq(2).all(), f"{col} : codes non 2 lettres"
    assert df_clean[col].str.isupper().all(), f"{col} : codes non majuscules"
    print(f"✔ {col} : {df_clean[col].nunique()} codes pays 2 lettres majuscules")

# Vérification : les devises sont 3 lettres majuscules (ISO 4217)
assert df_clean["salary_currency"].str.len().eq(3).all()
assert df_clean["salary_currency"].str.isupper().all()
print(f"✔ salary_currency : {df_clean['salary_currency'].nunique()} devises ISO 4217")

✔ experience_level : ['EN', 'EX', 'MI', 'SE']
✔ employment_type : ['CT', 'FL', 'FT', 'PT']
✔ company_size : ['L', 'M', 'S']
✔ remote_ratio : [np.int64(0), np.int64(50), np.int64(100)]
✔ employee_residence : 104 codes pays 2 lettres majuscules
✔ company_location : 97 codes pays 2 lettres majuscules
✔ salary_currency : 26 devises ISO 4217


In [14]:
# Cellule 4.4 : assertions sur les variables numériques
print("=== Vérifications numériques ===")

print(f"work_year min/max : {df_clean['work_year'].min()} / {df_clean['work_year'].max()}")
assert df_clean["work_year"].between(2020, 2025).all(), "work_year hors [2020, 2025]"
print("✔ work_year dans [2020, 2025]")

print(f"salary_in_usd min/max : {df_clean['salary_in_usd'].min()} / {df_clean['salary_in_usd'].max()}")
assert (df_clean["salary_in_usd"] > 0).all(), "salary_in_usd <= 0 détecté"
assert (df_clean["salary_in_usd"] <= 1_000_000).all(), "salary_in_usd > 1 000 000 détecté"
print("✔ salary_in_usd strictement positif et ≤ 1 000 000")

assert df_clean["remote_ratio"].isin([0, 50, 100]).all(), "remote_ratio hors {0, 50, 100}"
print("✔ remote_ratio ∈ {0, 50, 100}")

# Vérification : salary_in_usd entier (dtype int64)
assert df_clean["salary_in_usd"].dtype == "int64", f"dtype inattendu : {df_clean['salary_in_usd'].dtype}"
print("✔ salary_in_usd est bien int64")

=== Vérifications numériques ===
work_year min/max : 2020 / 2025
✔ work_year dans [2020, 2025]
salary_in_usd min/max : 15000 / 800000
✔ salary_in_usd strictement positif et ≤ 1 000 000
✔ remote_ratio ∈ {0, 50, 100}
✔ salary_in_usd est bien int64


In [15]:
# Cellule 4.5 : synthèse
print("=== Synthèse des vérifications ===")
print(f"Lignes : {len(df_clean)}")
print(f"Colonnes : {len(df_clean.columns)}")
print(f"Valeurs manquantes : {int(df_clean.isna().sum().sum())}")
print(f"Doublons complets (conservés) : {int(df_clean.duplicated().sum())}")
print()
print("Toutes les assertions sont passées. df_clean est conforme à la stratégie.")
print("Aucune transformation lourde n'a été appliquée (voir section 4.6).")

=== Synthèse des vérifications ===
Lignes : 151445
Colonnes : 11
Valeurs manquantes : 0
Doublons complets (conservés) : 79532

Toutes les assertions sont passées. df_clean est conforme à la stratégie.
Aucune transformation lourde n'a été appliquée (voir section 4.6).


### 4.6 Transformations non appliquées (et pourquoi)

Conformément à `docs/STRATEGIE_NETTOYAGE.md`, les transformations suivantes
ne sont **pas** appliquées à cette étape :

| Transformation | Décision | Justification |
|----------------|----------|---------------|
| Imputation | Non | 0 valeur manquante |
| Suppression des doublons dans le stockage | Non | Doublons légitimes (enquête multi-sources) ; déduplication au moment du split (étape 33) |
| Suppression des valeurs extrêmes | Non | Min 15k et max 800k sont des réalités métier |
| Conversion en `category` | Non | Déférée après le split train/test (étape 33) pour éviter toute fuite |
| Regroupement de `job_title` | Non | Déféré à l’étape 32 (`job_category`) |
| Transformation `log1p` de la cible | Non | À tester empiriquement à l’étape 32 |
| Exclusion de `salary` et `salary_currency` | Documentée, appliquée à l’étape 32 (liste blanche) | Fuite de données |

`df_clean` est donc **identique à `df` en contenu** (151 445 lignes, 11 colonnes,
0 valeur manquante, 79 532 doublons conservés). Il porte un nom distinct pour
signaler qu’il a passé les contrôles de qualité et qu’il sera la base des
analyses à venir. `df` reste disponible comme référence non modifiée.

## 5. Analyses par groupe

### 5.1 Objectif

Analyser les salaires par **niveau d'expérience**, **type d'emploi**, **taille
d'entreprise** et **télétravail**. Pour chaque groupe, on rapporte la **médiane**
et la **moyenne** : l'écart entre les deux révèle si l'asymétrie observée
globalement (skew 1,35) est une caractéristique de fond ou un artefact de
quelques valeurs extrêmes.

In [16]:
# Cellule 5.2 : salaire médian par niveau d'expérience
print("=== Salaire médian par niveau d'expérience ===")
median_by_exp = df_clean.groupby("experience_level")["salary_in_usd"].median().sort_values(ascending=False)
print(median_by_exp.map(lambda x: f"{x:,.0f} USD"))
print()

print("=== Effectifs par niveau ===")
print(df_clean["experience_level"].value_counts())
print()

print("=== Salaire médian + effectif dans un seul tableau ===")
synth_exp = (
    df_clean.groupby("experience_level")["salary_in_usd"]
    .agg(["count", "median", "mean"])
    .sort_values("median", ascending=False)
    .round(0)
)
print(synth_exp)

=== Salaire médian par niveau d'expérience ===
experience_level
EX    189,370 USD
SE    161,000 USD
MI    130,000 USD
EN     86,600 USD
Name: salary_in_usd, dtype: str

=== Effectifs par niveau ===
experience_level
SE    87491
MI    46128
EN    13663
EX     4163
Name: count, dtype: int64

=== Salaire médian + effectif dans un seul tableau ===
                  count    median      mean
experience_level                           
EX                 4163  189370.0  200471.0
SE                87491  161000.0  172658.0
MI                46128  130000.0  142301.0
EN                13663   86600.0   98962.0


In [17]:
# Cellule 5.3 : tableau complet par niveau d'expérience
print("=== Statistiques complètes par niveau d'expérience ===")
stats_exp = (
    df_clean.groupby("experience_level")["salary_in_usd"]
    .agg(
        effectif="count",
        moyenne="mean",
        mediane="median",
        q1=lambda x: x.quantile(0.25),
        q3=lambda x: x.quantile(0.75),
        ecart_type="std",
    )
    .round(0)
)
stats_exp["ecart_moy_med"] = (stats_exp["moyenne"] - stats_exp["mediane"]).round(0)
stats_exp["ratio_moy_med"] = (stats_exp["moyenne"] / stats_exp["mediane"]).round(3)
print(stats_exp)
print()
print("Si la colonne 'ecart_moy_med' est positive partout, la moyenne dépasse")
print("la médiane dans TOUS les groupes : l'asymétrie est structurelle.")

=== Statistiques complètes par niveau d'expérience ===
                  effectif   moyenne   mediane        q1        q3  ecart_type  ecart_moy_med  ratio_moy_med
experience_level                                                                                            
EN                   13663   98962.0   86600.0   62148.0  123900.0     55407.0        12362.0          1.143
EX                    4163  200471.0  189370.0  144600.0  249900.0     79458.0        11101.0          1.059
MI                   46128  142301.0  130000.0   92577.0  175800.0     72089.0        12301.0          1.095
SE                   87491  172658.0  161000.0  123000.0  210000.0     70876.0        11658.0          1.072

Si la colonne 'ecart_moy_med' est positive partout, la moyenne dépasse
la médiane dans TOUS les groupes : l'asymétrie est structurelle.


In [18]:
# Cellule 5.4 : autres variables catégorielles
variables = {
    "employment_type": "Type d'emploi",
    "company_size": "Taille d'entreprise",
    "remote_ratio": "Télétravail (%)",
}

for col, label in variables.items():
    print(f"\n=== {label} ({col}) ===")
    tableau = (
        df_clean.groupby(col)["salary_in_usd"]
        .agg(effectif="count", mediane="median", moyenne="mean")
        .sort_values("mediane", ascending=False)
        .round(0)
    )
    tableau["ecart_moy_med"] = (tableau["moyenne"] - tableau["mediane"]).round(0)
    print(tableau)


=== Type d'emploi (employment_type) ===
                 effectif   mediane   moyenne  ecart_moy_med
employment_type                                             
FT                 150541  147000.0  157931.0        10931.0
CT                    467  107712.0  104686.0        -3026.0
PT                    421   60000.0   76051.0        16051.0
FL                     16   50000.0   50652.0          652.0

=== Taille d'entreprise (company_size) ===
              effectif   mediane   moyenne  ecart_moy_med
company_size                                             
L                 3926  149800.0  159340.0         9540.0
M               147302  146400.0  157582.0        11182.0
S                  217   75000.0   87835.0        12835.0

=== Télétravail (%) (remote_ratio) ===
              effectif   mediane   moyenne  ecart_moy_med
remote_ratio                                             
0               119570  147000.0  159383.0        12383.0
100              31546  145000.0  151286.0   

In [19]:
# Cellule 5.5 : groupby multi-niveaux
print("=== Salaire médian par expérience × télétravail ===")
cross = (
    df_clean.groupby(["experience_level", "remote_ratio"])["salary_in_usd"]
    .agg(effectif="count", mediane="median")
    .round(0)
)
print(cross)
print()
print("Remarque : certaines combinaisons peuvent avoir des effectifs très faibles.")
print("On interprétera avec prudence (voir section 5.6).")

=== Salaire médian par expérience × télétravail ===
                               effectif   mediane
experience_level remote_ratio                    
EN               0                11514   88000.0
                 50                 106   50000.0
                 100               2043   84600.0
EX               0                 3297  188000.0
                 50                  11  125976.0
                 100                855  195804.0
MI               0                37955  132000.0
                 50                 123   65666.0
                 100               8050  120000.0
SE               0                66804  165000.0
                 50                  89   94000.0
                 100              20598  158100.0

Remarque : certaines combinaisons peuvent avoir des effectifs très faibles.
On interprétera avec prudence (voir section 5.6).


### 5.6 Interprétation

Les observations ci-dessous sont issues des tableaux produits dans les cellules
5.2 à 5.5. Chaque chiffre cité est vérifiable dans les sorties du notebook.

#### 1. Hiérarchie claire par niveau d'expérience

Les salaires médians suivent une progression stricte :

| Niveau | Médiane | Effectif |
|--------|---------|----------|
| EN | 86 600 USD | 13 663 |
| MI | 130 000 USD | 46 128 |
| SE | 161 000 USD | 87 491 |
| EX | 189 370 USD | 4 163 |

**Le rapport EX / EN est de 2,19×.** En revanche, les progressions relatives
ne sont pas uniformes : EN → MI (×1,50) est le gain le plus fort, puis MI → SE
(×1,24), puis SE → EX (×1,18). **Les rendements de l'expérience sont
décroissants** : le premier passage de junior à intermédiaire apporte le gain
relatif le plus important.

#### 2. Asymétrie structurelle (pas un artefact de valeurs extrêmes)

L'écart `moyenne − médiane` est **positif dans les 4 groupes** :

| Niveau | Écart moyen−médiane | Ratio moyen/médiane |
|--------|--------------------|---------------------|
| EN | +12 362 USD | 1,143 |
| MI | +12 301 USD | 1,095 |
| SE | +11 658 USD | 1,072 |
| EX | +11 101 USD | 1,059 |

L'asymétrie à droite observée globalement (skew 1,35) n'est **pas** due à
quelques valeurs extrêmes isolées : elle est **présente à l'intérieur de
chaque niveau**. En valeur absolue, l'écart est quasi constant (~11–12k USD),
mais **en valeur relative, il est le plus fort chez les juniors** (1,143 pour
EN contre 1,059 pour EX). Autrement dit, les salaires juniors sont plus
dispersés relativement à leur médiane que les salaires executives.

**Conséquence pour l'étape 32 :** la transformation `log1p` de la cible reste
pertinente. Elle pourrait réduire l'asymétrie de façon homogène sur tous les
groupes.

#### 3. Télétravail : pas de prime claire au full remote

| Télétravail | Médiane | Moyenne | Effectif |
|-------------|---------|---------|----------|
| 0 (présentiel) | 147 000 USD | 159 383 USD | 119 570 |
| 50 (hybride) | 69 999 USD | 81 712 USD | 329 |
| 100 (full remote) | 145 000 USD | 151 286 USD | 31 546 |

**Le présentiel et le full remote ont des médianes quasi identiques**
(147 000 vs 145 000, soit −1,4 %). Il n'y a donc **pas de prime salariale
au télétravail complet** dans ce dataset.

Le groupe hybride (`50`) affiche une médiane **anormalement basse** (69 999 USD).
Avec **seulement 329 observations** (0,2 % des lignes), cette valeur ne peut
pas être interprétée comme un effet du télétravail hybride. C'est probablement
un artefact : un petit segment spécifique (pays ou métier particulier) qui
tire la médiane vers le bas. **Ne pas conclure** sur cette base.

#### 4. Taille d'entreprise : L > M > S, mais S sous-représenté

| Taille | Médiane | Effectif |
|--------|---------|----------|
| L | 149 800 USD | 3 926 |
| M | 146 400 USD | 147 302 |
| S | 75 000 USD | 217 |

L'ordre L > M > S est **cohérent avec l'intuition métier**, mais deux limites :

- **M domine massivement** (147 302 / 151 445 = 97,3 %). L'échantillon est
  quasi mono-modal, ce qui limite le pouvoir discriminant de `company_size`.
- **S ne compte que 217 observations** (0,1 %). Sa médiane de 75 000 USD est
  frappante mais calculée sur un échantillon très faible. À interpréter avec
  prudence.

#### 5. Type d'emploi : `CT` est le seul groupe asymétrique à gauche

| Type | Médiane | Moyenne | Écart | Effectif |
|------|---------|---------|-------|----------|
| FT | 147 000 | 157 931 | +10 931 | 150 541 |
| CT | 107 712 | 104 686 | **−3 026** | 467 |
| PT | 60 000 | 76 051 | +16 051 | 421 |
| FL | 50 000 | 50 652 | +652 | 16 |

**`CT` (contrat) est le seul groupe avec une moyenne inférieure à la médiane**
(écart −3 026 USD). C'est une distribution asymétrique à **gauche** : quelques
contrats très bas tirent la moyenne vers le bas. Avec 467 observations, c'est
un signal à noter mais pas une conclusion robuste. Les autres groupes (FT, PT,
FL) présentent l'asymétrie à droite habituelle.

`FL` (freelance) n'a que 16 observations : aucune interprétation n'est
possible.

#### 6. Croisement expérience × télétravail : effet d'interaction réel

Extraits du croisement (effectifs et médianes) :

| Niveau | Présentiel (0) | Hybride (50) | Full remote (100) |
|--------|----------------|--------------|-------------------|
| EN | 88 000 (11 514) | 50 000 (106) | 84 600 (2 043) |
| MI | 132 000 (37 955) | 65 666 (123) | 120 000 (8 050) |
| SE | 165 000 (66 804) | 94 000 (89) | 158 100 (20 598) |
| EX | 188 000 (3 297) | 125 976 (11) | 195 804 (855) |

**Le full remote n'est avantageux que pour les executives** (195 804 vs 188 000,
soit +4,2 %). Pour les niveaux EN, MI et SE, le présentiel est **légèrement
mieux payé** que le full remote. C'est un **effet d'interaction** :
l'effet du télétravail dépend du niveau d'expérience.

**Le groupe hybride (50) est systématiquement très mal payé** (50k à 126k selon
le niveau), mais ses effectifs sont minuscules (11 à 123 par cellule). Cette
observation ne peut pas être généralisée sans investigation supplémentaire.
Elle pourrait refléter un segment spécifique du dataset (par exemple un pays
où l'hybride est fréquent et les salaires bas).

#### 7. Rappel méthodologique

Ces analyses montrent des **associations**, pas des relations **causales**.
Un salaire plus élevé chez les seniors peut refléter l'expérience, mais aussi
la sélection (les seniors sont sur-représentés dans certains pays, certains
métiers, certaines tailles d'entreprise). Le croisement expérience × télétravail
suggère un effet d'interaction, mais la modélisation (phase 6) sera nécessaire
pour **isoler** l'effet de chaque variable, toutes choses égales par ailleurs.
La causalité ne pourra jamais être démontrée sans design expérimental.

#### 8. Décisions pour la suite

- `experience_level` est la variable la plus discriminante observée → à
  conserver en priorité dans le modèle.
- `remote_ratio` n'a pas d'effet simple → à conserver mais avec prudence
  (effet d'interaction possible avec `experience_level`).
- `company_size` est quasi constant (`M` à 97,3 %) → faible pouvoir
  discriminant attendu, à documenter dans l'analyse d'importance des variables.
- `employment_type` est quasi constant (`FT` à 99,4 %) → même conclusion.
- Le groupe hybride (`remote_ratio=50`, 329 observations) mérite une
  investigation à l'étape 22 (répartition par pays) pour comprendre la médiane
  anormalement basse.

### 5.7 Analyse par pays et par métier

On analyse maintenant les deux dimensions qui déterminent le plus le salaire :
le **pays** (résidence de l'employé) et l'**intitulé du poste**.
Les groupes à faible effectif sont systématiquement signalés : une médiane
calculée sur moins de 30 observations est instable.

Le référentiel pays → continent et le regroupement des intitulés de poste
seront traités aux étapes 22 et 32.

In [20]:
# Cellule 5.8 : top 20 pays par effectif
print("=== Top 20 pays par nombre d'employés (employee_residence) ===")
top_pays = (
    df_clean.groupby("employee_residence")["salary_in_usd"]
    .agg(effectif="count", mediane="median", moyenne="mean")
    .sort_values("effectif", ascending=False)
    .head(20)
    .round(0)
)
top_pays["ecart_moy_med"] = (top_pays["moyenne"] - top_pays["mediane"]).round(0)
print(top_pays)
print()

print(f"Nombre total de pays (résidence) : {df_clean['employee_residence'].nunique()}")
print(f"Pays couvrant 90 % des lignes : ", end="")
cumul = df_clean["employee_residence"].value_counts().cumsum() / len(df_clean)
print(int((cumul < 0.90).sum()) + 1)

=== Top 20 pays par nombre d'employés (employee_residence) ===
                    effectif   mediane   moyenne  ecart_moy_med
employee_residence                                             
US                    135506  151270.0  163945.0        12675.0
CA                      6207  120000.0  129644.0         9644.0
GB                      4172   75522.0   95072.0        19550.0
AU                       590  117618.0  128508.0        10890.0
NL                       482   72000.0   78366.0         6366.0
DE                       445   88444.0  105016.0        16572.0
FR                       441   56842.0   69062.0        12220.0
LT                       372   54358.0   59626.0         5268.0
AT                       355   50866.0   58457.0         7591.0
ES                       313   48585.0   64773.0        16188.0
IN                       260   38884.0   59189.0        20305.0
SK                       248   31578.0   36202.0         4624.0
PL                       168   73855.0   

In [21]:
# Cellule 5.9 : pays extrêmes en médiane, avec effectif suffisant
pays_stats = (
    df_clean.groupby("employee_residence")["salary_in_usd"]
    .agg(effectif="count", mediane="median")
    .query("effectif >= 100")
    .sort_values("mediane", ascending=False)
)

print("=== Top 10 pays les mieux payés (médiane, effectif ≥ 100) ===")
print(pays_stats.head(10).round(0))
print()

print("=== Top 10 pays les moins bien payés (médiane, effectif ≥ 100) ===")
print(pays_stats.tail(10).round(0))
print()

print(f"Nombre de pays avec effectif ≥ 100 : {len(pays_stats)}")

=== Top 10 pays les mieux payés (médiane, effectif ≥ 100) ===
                    effectif   mediane
employee_residence                    
US                    135506  151270.0
CA                      6207  120000.0
AU                       590  117618.0
IE                       149  100000.0
DE                       445   88444.0
GB                      4172   75522.0
PL                       168   73855.0
NL                       482   72000.0
BR                       114   60000.0
FR                       441   56842.0

=== Top 10 pays les moins bien payés (médiane, effectif ≥ 100) ===
                    effectif  mediane
employee_residence                   
NL                       482  72000.0
BR                       114  60000.0
FR                       441  56842.0
LT                       372  54358.0
AT                       355  50866.0
ES                       313  48585.0
MX                       112  48000.0
LV                       118  41642.0
IN                    

In [22]:
# Cellule 5.10 : top 20 intitulés de poste par effectif
print("=== Top 20 intitulés de poste par effectif ===")
top_jobs = (
    df_clean.groupby("job_title")["salary_in_usd"]
    .agg(effectif="count", mediane="median", moyenne="mean")
    .sort_values("effectif", ascending=False)
    .head(20)
    .round(0)
)
top_jobs["ecart_moy_med"] = (top_jobs["moyenne"] - top_jobs["mediane"]).round(0)
print(top_jobs)
print()

print(f"Nombre total d'intitulés distincts : {df_clean['job_title'].nunique()}")
print(f"Intitulés couvrant 80 % des lignes : ", end="")
cumul = df_clean["job_title"].value_counts().cumsum() / len(df_clean)
n_jobs_80 = int((cumul < 0.80).sum()) + 1
print(n_jobs_80)
print()

print("=== 10 intitulés les moins fréquents ===")
print(df_clean["job_title"].value_counts().tail(10))

=== Top 20 intitulés de poste par effectif ===
                           effectif   mediane   moyenne  ecart_moy_med
job_title                                                             
Data Scientist                18751  150000.0  156883.0         6883.0
Software Engineer             16948  180000.0  190122.0        10122.0
Data Engineer                 16352  140000.0  147426.0         7426.0
Data Analyst                  13779   99000.0  105169.0         6169.0
Engineer                      11004  159000.0  170669.0        11669.0
Machine Learning Engineer      8887  189000.0  198707.0         9707.0
Manager                        7811  155500.0  166479.0        10979.0
Analyst                        5396  105000.0  112758.0         7758.0
Research Scientist             3460  180000.0  197515.0        17515.0
Product Manager                2576  175000.0  189526.0        14526.0
Applied Scientist              2381  163000.0  183686.0        20686.0
Associate                     

In [23]:
# Cellule 5.11 : croisement expérience × top 5 métiers
top5_jobs = top_jobs.head(5).index.tolist()
print(f"Top 5 métiers retenus : {top5_jobs}")

subset = df_clean[df_clean["job_title"].isin(top5_jobs)]

croise = (
    subset.groupby(["job_title", "experience_level"])["salary_in_usd"]
    .agg(effectif="count", mediane="median")
    .round(0)
)
print("\n=== Salaire médian par métier × niveau d'expérience ===")
print(croise)
print()
print("Remarque : les cellules avec effectif < 30 doivent être interprétées")
print("avec prudence (intervalle de confiance large).")

Top 5 métiers retenus : ['Data Scientist', 'Software Engineer', 'Data Engineer', 'Data Analyst', 'Engineer']

=== Salaire médian par métier × niveau d'expérience ===
                                    effectif   mediane
job_title         experience_level                    
Data Analyst      EN                    5227   81453.0
                  EX                     110  114420.0
                  MI                    2600   90000.0
                  SE                    5842  119555.0
Data Engineer     EN                     655   81000.0
                  EX                     577  194500.0
                  MI                    5242  120000.0
                  SE                    9878  153000.0
Data Scientist    EN                     829   96428.0
                  EX                     285  189370.0
                  MI                    5786  128000.0
                  SE                   11851  160000.0
Engineer          EN                     550   95000.0
         

## 6. Référentiel pays → continent et fusion

### 6.1 Objectif

Les 104 pays de `employee_residence` ont des effectifs très hétérogènes : seuls
17 dépassent 100 observations. Agréger par continent (7 zones) donne des
groupes plus robustes et permet de tester si l'effet « US = mieux payé » se
retrouve au niveau continental.

On crée un référentiel `COUNTRY_TO_CONTINENT` (dict Python), on vérifie
qu'aucun code pays n'est non mappé, puis on fusionne avec `merge()`.

In [24]:
# Cellule 6.2 : référentiel pays -> continent
# Classification inspirée de la nomenclature M49 de l'ONU, adaptée pour
# regrouper les zones géographiques pertinentes pour l'analyse salariale.
COUNTRY_TO_CONTINENT = {
    # Amérique du Nord
    "US": "North America", "CA": "North America", "MX": "North America",
    "BM": "North America", "PR": "North America",
    # Amérique du Sud
    "BR": "South America", "AR": "South America", "CL": "South America",
    "CO": "South America", "PE": "South America", "VE": "South America",
    "UY": "South America", "EC": "South America", "BO": "South America",
    "PY": "South America", "CR": "South America", "PA": "South America",
    "DO": "South America", "GT": "South America", "SV": "South America",
    "HN": "South America", "NI": "South America", "CU": "South America",
    "JM": "South America", "TT": "South America", "BB": "South America",
    "BS": "South America", "HT": "South America",
    # Europe
    "GB": "Europe", "DE": "Europe", "FR": "Europe", "NL": "Europe",
    "ES": "Europe", "IT": "Europe", "IE": "Europe", "PT": "Europe",
    "PL": "Europe", "LT": "Europe", "LV": "Europe", "AT": "Europe",
    "SK": "Europe", "FI": "Europe", "SE": "Europe", "NO": "Europe",
    "DK": "Europe", "CH": "Europe", "BE": "Europe", "CZ": "Europe",
    "HU": "Europe", "RO": "Europe", "BG": "Europe", "GR": "Europe",
    "HR": "Europe", "SI": "Europe", "EE": "Europe", "LU": "Europe",
    "IS": "Europe", "MT": "Europe", "CY": "Europe", "RS": "Europe",
    "UA": "Europe", "BY": "Europe", "MD": "Europe", "AL": "Europe",
    "MK": "Europe", "BA": "Europe", "ME": "Europe", "RU": "Europe",
    "AD": "Europe", "MC": "Europe", "SM": "Europe", "LI": "Europe",
    "GI": "Europe", "FO": "Europe", "GL": "Europe",
    # Asie
    "IN": "Asia", "JP": "Asia", "CN": "Asia", "SG": "Asia",
    "HK": "Asia", "KR": "Asia", "TW": "Asia", "MY": "Asia",
    "TH": "Asia", "ID": "Asia", "PH": "Asia", "VN": "Asia",
    "PK": "Asia", "BD": "Asia", "LK": "Asia", "NP": "Asia",
    "KH": "Asia", "MM": "Asia", "KZ": "Asia", "UZ": "Asia",
    "GE": "Asia", "AM": "Asia", "AZ": "Asia", "MN": "Asia",
    "BN": "Asia", "MO": "Asia", "KG": "Asia", "TJ": "Asia",
    "TM": "Asia", "AF": "Asia", "BT": "Asia", "MV": "Asia",
    # Moyen-Orient
    "AE": "Middle East", "SA": "Middle East", "QA": "Middle East",
    "KW": "Middle East", "BH": "Middle East", "OM": "Middle East",
    "JO": "Middle East", "LB": "Middle East", "IL": "Middle East",
    "TR": "Middle East", "IR": "Middle East", "IQ": "Middle East",
    "SY": "Middle East", "YE": "Middle East", "PS": "Middle East",
    # Afrique
    "ZA": "Africa", "EG": "Africa", "NG": "Africa", "KE": "Africa",
    "MA": "Africa", "TN": "Africa", "GH": "Africa", "DZ": "Africa",
    "ET": "Africa", "UG": "Africa", "TZ": "Africa", "RW": "Africa",
    "SN": "Africa", "CI": "Africa", "CM": "Africa", "ZW": "Africa",
    "ZM": "Africa", "MU": "Africa", "MG": "Africa", "AO": "Africa",
    "MZ": "Africa", "BW": "Africa", "NA": "Africa", "MW": "Africa",
    "LY": "Africa", "SD": "Africa", "CD": "Africa", "BF": "Africa",
    "ML": "Africa", "NE": "Africa", "TD": "Africa", "SO": "Africa",
    "ER": "Africa", "DJ": "Africa", "GM": "Africa", "GN": "Africa",
    "SL": "Africa", "LR": "Africa", "TG": "Africa", "BJ": "Africa",
    "GA": "Africa", "CG": "Africa", "CF": "Africa", "BI": "Africa",
    "LS": "Africa", "SZ": "Africa", "SC": "Africa", "CV": "Africa",
    "MR": "Africa", "ST": "Africa", "GQ": "Africa", "KM": "Africa",
    # Océanie
    "AU": "Oceania", "NZ": "Oceania", "FJ": "Oceania", "PG": "Oceania",
    "NC": "Oceania", "PF": "Oceania", "WS": "Oceania", "TO": "Oceania",
    "VU": "Oceania", "SB": "Oceania", "GU": "Oceania",
        # Codes ajoutés après diagnostic des non-mappés
    "AS": "Oceania",       # American Samoa
    "JE": "Europe",        # Jersey
    "XK": "Europe",        # Kosovo (code non ISO, utilisé par le dataset)
}

print(f"Nombre de pays dans le référentiel : {len(COUNTRY_TO_CONTINENT)}")
print(f"Nombre de continents distincts : {len(set(COUNTRY_TO_CONTINENT.values()))}")
print(f"Continents : {sorted(set(COUNTRY_TO_CONTINENT.values()))}")

Nombre de pays dans le référentiel : 188
Nombre de continents distincts : 7
Continents : ['Africa', 'Asia', 'Europe', 'Middle East', 'North America', 'Oceania', 'South America']


In [25]:
# Cellule de réinitialisation : à exécuter avant 6.3
df_clean = df.copy()
print("df_clean réinitialisé :", df_clean.shape)

df_clean réinitialisé : (151445, 11)


In [26]:
# Cellule 6.3 (corrigée) : vérification d'intégrité + merge idempotent
print("=== Vérification : aucun code pays non mappé ===")

codes_residence = set(df_clean["employee_residence"].unique())
codes_location  = set(df_clean["company_location"].unique())
codes_union     = codes_residence | codes_location

print(f"Codes uniques employee_residence : {len(codes_residence)}")
print(f"Codes uniques company_location   : {len(codes_location)}")
print(f"Union des codes                  : {len(codes_union)}")

non_mappes = sorted(codes_union - COUNTRY_TO_CONTINENT.keys())
print(f"Codes non mappés                 : {len(non_mappes)}")
assert not non_mappes, f"Codes non mappés à ajouter au référentiel : {non_mappes}"

# Idempotence : si les colonnes existent déjà (réexécution), on les supprime
cols_a_supprimer = [
    c for c in ["residence_continent", "company_continent"]
    if c in df_clean.columns
]
if cols_a_supprimer:
    df_clean = df_clean.drop(columns=cols_a_supprimer)
    print(f"Colonnes existantes supprimées (réexécution) : {cols_a_supprimer}")

df_continent = pd.DataFrame(
    list(COUNTRY_TO_CONTINENT.items()),
    columns=["country_code", "continent"],
)
print(f"\nDataFrame de référence : {df_continent.shape}")

df_clean = df_clean.merge(
    df_continent.rename(columns={"continent": "residence_continent"}),
    left_on="employee_residence",
    right_on="country_code",
    how="left",
).drop(columns="country_code")

df_clean = df_clean.merge(
    df_continent.rename(columns={"continent": "company_continent"}),
    left_on="company_location",
    right_on="country_code",
    how="left",
).drop(columns="country_code")

print(f"\nAprès fusion, shape de df_clean : {df_clean.shape}")
print(f"Valeurs manquantes residence_continent : {df_clean['residence_continent'].isna().sum()}")
print(f"Valeurs manquantes company_continent   : {df_clean['company_continent'].isna().sum()}")

assert df_clean.shape == (151445, 13), f"Shape inattendue : {df_clean.shape}"
assert df_clean["residence_continent"].notna().all()
assert df_clean["company_continent"].notna().all()
print("✔ Merge OK : 151 445 lignes, 0 continent manquant, 13 colonnes")

=== Vérification : aucun code pays non mappé ===
Codes uniques employee_residence : 104
Codes uniques company_location   : 97
Union des codes                  : 106
Codes non mappés                 : 0

DataFrame de référence : (188, 2)

Après fusion, shape de df_clean : (151445, 13)
Valeurs manquantes residence_continent : 0
Valeurs manquantes company_continent   : 0
✔ Merge OK : 151 445 lignes, 0 continent manquant, 13 colonnes


In [27]:
# Cellule 6.4 : analyse par continent
print("=== Salaires par continent de résidence ===")
par_continent = (
    df_clean.groupby("residence_continent")["salary_in_usd"]
    .agg(effectif="count", mediane="median", moyenne="mean")
    .sort_values("mediane", ascending=False)
    .round(0)
)
par_continent["ecart_moy_med"] = (
    par_continent["moyenne"] - par_continent["mediane"]
).round(0)
par_continent["pct_lignes"] = (
    par_continent["effectif"] / len(df_clean) * 100
).round(1)
print(par_continent)
print()

print("=== Salaires par continent de l'employeur ===")
par_continent_emp = (
    df_clean.groupby("company_continent")["salary_in_usd"]
    .agg(effectif="count", mediane="median")
    .sort_values("mediane", ascending=False)
    .round(0)
)
print(par_continent_emp)
print()

# Croisement : continent de résidence × niveau d'expérience
print("=== Salaire médian par continent × niveau d'expérience ===")
cross_continent = (
    df_clean.groupby(["residence_continent", "experience_level"])["salary_in_usd"]
    .agg(effectif="count", mediane="median")
    .round(0)
)
print(cross_continent)

=== Salaires par continent de résidence ===
                     effectif   mediane   moyenne  ecart_moy_med  pct_lignes
residence_continent                                                         
North America          141839  150000.0  162374.0        12374.0        93.7
Oceania                   671  117718.0  127027.0         9309.0         0.4
Middle East                76   99200.0  102974.0         3774.0         0.1
Europe                   7864   68421.0   84700.0        16279.0         5.2
Africa                    169   60000.0   80324.0        20324.0         0.1
South America             335   58200.0   65504.0         7304.0         0.2
Asia                      491   41385.0   63423.0        22038.0         0.3

=== Salaires par continent de l'employeur ===
                   effectif   mediane
company_continent                    
North America        141902  150000.0
Oceania                 679  117363.0
Middle East              75  100000.0
Europe                 785

### 6.5 Interprétation



1. **Hiérarchie continentale** : l'Amérique du Nord domine-t-elle en médiane ?
   L'Europe et l'Océanie suivent-elles ? Où se situent l'Asie, le Moyen-Orient,
   l'Amérique du Sud et l'Afrique ?
2. **Effet de la sur-représentation US** : l'Amérique du Nord est-elle
   quasi exclusivement les États-Unis (89,5 % des lignes totales) ? Si oui,
   la médiane continentale est essentiellement une médiane US.
3. **Écart résidence vs employeur** : y a-t-il des continents où la médiane
   `residence_continent` diffère nettement de `company_continent` ?
   (Cas de travail à distance transfrontalier : résidence dans un pays,
   employeur dans un autre.)
4. **Croisement continent × expérience** : la hiérarchie des niveaux
   (EN < MI < SE < EX) est-elle respectée dans **tous** les continents ?
   Si oui, l'effet de l'expérience est robuste au contexte géographique.
5. **Effectifs faibles** : quels continents ont moins de 100 observations ?
   Leurs médianes doivent être interprétées avec prudence.

**Rappel méthodologique :** ces analyses montrent des associations. Le
continent n'est pas une cause du salaire : il regroupe des pays avec des
marchés du travail, des coûts de la vie et des devises différents. La
modélisation (phase 6) isolera l'effet propre de chaque variable.

## 7. Tableaux croisés avec `pivot_table()`

### 7.1 Objectif

`pivot_table()` produit des tableaux à **deux dimensions** : lignes = une variable,
colonnes = une autre, valeurs = une agrégation (médiane, effectif, etc.).
C'est l'outil adapté pour comparer des combinaisons de catégories.

On construit deux tableaux croisés :

1. **`experience_level × residence_continent`** : salaire médian et effectifs.
2. **`job_title (top 10) × experience_level`** : salaire médian pour les métiers
   les plus représentés.

On ajoute les **effectifs** dans un tableau séparé pour ne jamais interpréter
une médiane sans savoir sur combien d'observations elle est calculée.

In [28]:
# Cellule 7.2 : pivot expérience × continent — médiane et effectifs
# On construit deux tableaux : médiane et effectif, pour les lire côte à côte.

pivot_median = df_clean.pivot_table(
    values="salary_in_usd",
    index="experience_level",
    columns="residence_continent",
    aggfunc="median",
    fill_value=0,          # cellules sans observation -> 0 (à lire comme "aucun cas")
    margins=True,          # ajoute une ligne/colonne "All" (moyenne globale des médianes)
    margins_name="Tous",
)

pivot_count = df_clean.pivot_table(
    values="salary_in_usd",
    index="experience_level",
    columns="residence_continent",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Tous",
)

print("=== Salaire MÉDIAN (USD) par niveau d'expérience × continent ===")
print(pivot_median.round(0).astype(int))
print()
print("=== EFFECTIFS par niveau d'expérience × continent ===")
print(pivot_count.astype(int))
print()
print("⚠ Les cellules avec effectif < 30 ne doivent pas être interprétées")
print("  comme des différences significatives (intervalle de confiance large).")

=== Salaire MÉDIAN (USD) par niveau d'expérience × continent ===
residence_continent  Africa    Asia  Europe  Middle East  North America  Oceania  South America    Tous
experience_level                                                                                       
EN                    40290   28649   48138        54250          93274   106025          30000   86600
EX                   250000  150000  139240            0         191000   228292         110000  189370
MI                    60000   41608   63098        95000         135000   113812          48000  130000
SE                    66642   51605   82555       105000         165000   132475          72000  161000
Tous                  60000   41385   68421        99200         150000   117718          58200  146100

=== EFFECTIFS par niveau d'expérience × continent ===
residence_continent  Africa  Asia  Europe  Middle East  North America  Oceania  South America    Tous
experience_level                                  

In [29]:
# Cellule 7.3 : pivot top 10 métiers × niveau d'expérience
top10_jobs = (
    df_clean["job_title"].value_counts().head(10).index.tolist()
)
print(f"Top 10 métiers retenus : {top10_jobs}")
print()

subset_top10 = df_clean[df_clean["job_title"].isin(top10_jobs)]

pivot_jobs_median = subset_top10.pivot_table(
    values="salary_in_usd",
    index="job_title",
    columns="experience_level",
    aggfunc="median",
    fill_value=0,
    margins=True,
    margins_name="Tous",
)

pivot_jobs_count = subset_top10.pivot_table(
    values="salary_in_usd",
    index="job_title",
    columns="experience_level",
    aggfunc="count",
    fill_value=0,
    margins=True,
    margins_name="Tous",
)

print("=== Salaire MÉDIAN (USD) par métier (top 10) × niveau d'expérience ===")
print(pivot_jobs_median.round(0).astype(int))
print()
print("=== EFFECTIFS par métier (top 10) × niveau d'expérience ===")
print(pivot_jobs_count.astype(int))
print()
print("⚠ Les cellules avec effectif < 30 (notamment EX) sont peu fiables.")

Top 10 métiers retenus : ['Data Scientist', 'Software Engineer', 'Data Engineer', 'Data Analyst', 'Engineer', 'Machine Learning Engineer', 'Manager', 'Analyst', 'Research Scientist', 'Product Manager']

=== Salaire MÉDIAN (USD) par métier (top 10) × niveau d'expérience ===
experience_level               EN      EX      MI      SE    Tous
job_title                                                        
Analyst                     86600  144300   99355  115800  105000
Data Analyst                81453  114420   90000  119555   99000
Data Engineer               81000  194500  120000  153000  140000
Data Scientist              96428  189370  128000  160000  150000
Engineer                    95000  232500  133590  170000  159000
Machine Learning Engineer  139650  220000  165000  198200  189000
Manager                    127402  176720  140175  171600  155500
Product Manager            127800  195000  165120  180000  175000
Research Scientist         177000  185000  173000  195200  180000


In [30]:
# Cellule 7.4 : part de chaque niveau d'expérience dans chaque continent
# Utile pour voir si la structure des niveaux varie selon le continent.

# Pourcentage de chaque niveau d'expérience au sein de chaque continent
pct_within_continent = df_clean.pivot_table(
    values="salary_in_usd",
    index="residence_continent",
    columns="experience_level",
    aggfunc="count",
    fill_value=0,
)
# Normalisation par ligne : chaque ligne somme à 100 %
pct_within_continent = pct_within_continent.div(
    pct_within_continent.sum(axis=1), axis=0
) * 100

print("=== Répartition des niveaux d'expérience par continent (% en ligne) ===")
print(pct_within_continent.round(1))
print()

# Agrégation multiple : médiane et effectif dans un seul pivot multi-niveaux
pivot_multi = df_clean.pivot_table(
    values="salary_in_usd",
    index="residence_continent",
    columns="experience_level",
    aggfunc=["median", "count"],
    fill_value=0,
)
print("=== Pivot multi-agrégations (median + count) ===")
print(pivot_multi.round(0).astype(int))

=== Répartition des niveaux d'expérience par continent (% en ligne) ===
experience_level       EN   EX    MI    SE
residence_continent                       
Africa               14.2  1.8  41.4  42.6
Asia                 20.6  1.8  38.9  38.7
Europe               20.3  2.5  36.3  40.9
Middle East          10.5  0.0  35.5  53.9
North America         8.3  2.8  30.0  58.9
Oceania              22.1  1.2  41.4  35.3
South America        23.6  0.9  31.9  43.6

=== Pivot multi-agrégations (median + count) ===
                     median                          count                    
experience_level         EN      EX      MI      SE     EN    EX     MI     SE
residence_continent                                                           
Africa                40290  250000   60000   66642     24     3     70     72
Asia                  28649  150000   41608   51605    101     9    191    190
Europe                48138  139240   63098   82555   1599   194   2856   3215
Middle East      

### 7.5 Interprétation

Les observations ci-dessous sont issues des tableaux produits dans les cellules
7.2 à 7.4. Chaque chiffre cité est vérifiable dans les sorties du notebook.

#### 1. Structure des niveaux par continent : North America est le continent le plus senior

| Continent | % EN | % MI | % SE | % EX | Médiane globale |
|-----------|------|------|------|------|-----------------|
| North America | 8,3 | 30,0 | **58,9** | 2,8 | 150 000 |
| Europe | 20,3 | 36,3 | 40,9 | 2,5 | 68 421 |
| Oceania | 22,1 | 41,4 | 35,3 | 1,2 | 117 718 |
| Asia | 20,6 | 38,9 | 38,7 | 1,8 | 41 385 |
| South America | **23,6** | 31,9 | 43,6 | 0,9 | 58 200 |
| Africa | 14,2 | **41,4** | 42,6 | 1,8 | 60 000 |
| Middle East | 10,5 | 35,5 | 53,9 | **0,0** | 99 200 |

**L'Amérique du Nord est le continent le plus senior** : 58,9 % de `SE` et seulement 8,3 % de `EN`, contre 20,3 % de `EN` en Europe et 22,1 % en Océanie. La médiane élevée de l'Amérique du Nord (150 000 USD) reflète donc **en partie** un effet de composition : plus de seniors, moins de juniors.

**L'Océanie et l'Amérique du Sud ont la plus forte proportion de juniors** (22,1 % et 23,6 %), mais l'Océanie paie très bien ses juniors (EN à 106 025 USD, contre 30 000 en Amérique du Sud). C'est un signal que la structure d'expérience n'explique **pas tout** : à niveau égal, les salaires océaniens restent élevés.

**Le Middle East n'a aucun `EX`** dans les données (effectif 0). C'est une limite structurelle du dataset pour ce continent : aucune analyse executives n'est possible.

#### 2. À niveau d'expérience égal, North America domine largement

| Niveau | North America | Europe | Ratio NA / Europe |
|--------|---------------|--------|-------------------|
| EN | 93 274 | 48 138 | **1,94×** |
| MI | 135 000 | 63 098 | **2,14×** |
| SE | 165 000 | 82 555 | **2,00×** |
| EX | 191 000 | 139 240 | **1,37×** |

**L'écart North America / Europe persiste à niveau égal** : il est de ~2× pour EN, MI et SE, et de 1,37× pour EX. Cela signifie que **l'écart global (150 000 vs 68 421, soit 2,19×) n'est pas uniquement un effet de composition**. Même à niveau d'expérience égal, un professionnel nord-américain gagne environ **deux fois plus** qu'un européen dans ce dataset.

**Conséquence pour la modélisation :** `residence_continent` et `experience_level` ont un effet **additif et interactif** — leur combinaison apporte plus d'information que chaque variable prise séparément. Les modèles arborescents (Random Forest, Gradient Boosting) captureront cette interaction ; une régression linéaire simple devra peut-être inclure un terme d'interaction.

#### 3. Progressions avec l'expérience : fortes disparités selon les continents

| Continent | Ratio EN→SE | Ratio EN→EX | Ratio SE→EX |
|-----------|-------------|-------------|-------------|
| North America | 1,77× | 2,05× | 1,16× |
| Europe | 1,72× | **2,89×** | 1,69× |
| Asia | 1,80× | 5,24× (⚠ EX = 9) | 2,91× (⚠) |
| South America | 2,40× | 3,67× (⚠ EX = 3) | 1,53× |
| Oceania | 1,25× | 2,15× (⚠ EX = 8) | 1,72× |
| Africa | 1,65× | 6,20× (⚠ EX = 3) | 3,75× (⚠) |

**L'Europe a la progression `EN → EX` la plus forte parmi les continents à effectifs suffisants** (2,89×), tandis que l'Amérique du Nord a une progression plus modérée (2,05×). Cela suggère que le marché européen **récompense davantage l'accession au niveau executive** en termes relatifs.

Les ratios asiatiques, africains et sud-américains sont très élevés mais reposent sur **moins de 10 observations EX** chacun. **Ne pas conclure** : ces valeurs sont indicatives, non robustes.

#### 4. Métiers : les progressions relatives sont très différentes

**Ratio EN→SE par métier** (calculé à partir du pivot 7.3) :

| Métier | EN | SE | Ratio SE/EN |
|--------|-----|-----|-------------|
| Data Engineer | 81 000 | 153 000 | **1,89×** |
| Engineer | 95 000 | 170 000 | **1,79×** |
| Data Scientist | 96 428 | 160 000 | **1,66×** |
| Software Engineer | 115 200 | 190 000 | **1,65×** |
| Data Analyst | 81 453 | 119 555 | 1,47× |
| ML Engineer | 139 650 | 198 200 | 1,42× |
| Product Manager | 127 800 | 180 000 | 1,41× |
| Manager | 127 402 | 171 600 | 1,35× |
| Analyst | 86 600 | 115 800 | 1,34× |
| Research Scientist | 177 000 | 195 200 | **1,10×** |

**Deux profils opposés :**

- **Data Engineer, Engineer, Data Scientist, Software Engineer** : progression **forte** (1,65× à 1,89×). Ces métiers « grimpent » nettement avec l'expérience.
- **Research Scientist, Analyst, Manager, Product Manager** : progression **faible** (1,10× à 1,41×). Ces métiers sont **déjà bien payés à l'entrée** (Research Scientist 177 000 en EN, Manager 127 402, Product Manager 127 800), mais progressent peu en relatif.

**Interprétation métier :** les métiers techniques (Data Engineer, Engineer, SWE) offrent une **trajectoire salariale ascendante** avec l'expérience, tandis que les métiers de recherche et de management sont **déjà valorisés dès l'entrée**. C'est une information utile pour un candidat qui choisit une orientation : un junior en Research Scientist gagnera plus qu'un junior en Data Engineer, mais un senior en Data Engineer (153 000) gagnera presque autant qu'un senior en Research Scientist (195 200 → l'écart se réduit).

#### 5. Métiers avec un `EN` déjà très élevé

Cinq métiers ont une médiane `EN` supérieure à 110 000 USD :

| Métier | Médiane EN | Effectif EN |
|--------|-----------|-------------|
| Research Scientist | 177 000 | 365 |
| ML Engineer | 139 650 | 416 |
| Product Manager | 127 800 | **30** ⚠ |
| Manager | 127 402 | 62 |
| Software Engineer | 115 200 | 626 |

**Attention :** Product Manager `EN` n'a que **30 observations** — c'est à la limite du seuil de fiabilité. Les autres (Research Scientist, ML Engineer, SWE) ont des effectifs suffisants (365, 416, 626) pour être interprétés.

**Conséquence :** ces métiers recrutent des juniors à des salaires déjà élevés, soit parce qu'ils exigent des compétences rares (Research Scientist, ML Engineer), soit parce que le titre « Software Engineer » est souvent porté par des profils expérimentés même en début de carrière dans ce dataset.

#### 6. Effectifs faibles : liste des cellules à exclure des conclusions

**Dans le pivot métier × expérience :**

- **`EX` de Research Scientist : 3 observations** (médiane 185 000). À exclure.
- **`EX` de Product Manager : 30 observations** (médiane 195 000). À la limite.
- **`EN` de Product Manager : 30 observations** (médiane 127 800). À la limite.
- **`EN` de Manager : 62 observations** (médiane 127 402). Faible mais utilisable avec prudence.
- **`EX` de Manager : 132 observations**, `EX` de Analyst : 78, `EX` de ML Engineer : 76 — utilisables mais à signaler.

**Dans le pivot expérience × continent :**

- **`EX` de Africa (3), Asia (9), South America (3), Oceania (8)** : à exclure.
- **`EN` de Middle East (8), `MI` de Middle East (27)** : à exclure.
- **`EX` de Middle East (0)** : aucune observation.

**Convention appliquée :** toute cellule avec effectif < 30 est signalée comme non fiable. Toute cellule entre 30 et 100 est utilisable mais avec prudence. Seules les cellules ≥ 100 sont pleinement robustes.

#### 7. Contrôle de cohérence avec les analyses précédentes

Les colonnes « Tous » des pivots doivent retrouver les médianes globales calculées à l'étape 21 (cellule 5.10) :

| Métier | Médiane 7.3 | Médiane 5.10 | Écart |
|--------|-------------|--------------|-------|
| Data Scientist | 150 000 | 150 000 | 0 |
| Software Engineer | 180 000 | 180 000 | 0 |
| Data Engineer | 140 000 | 140 000 | 0 |
| Data Analyst | 99 000 | 99 000 | 0 |
| ML Engineer | 189 000 | 189 000 | 0 |
| Manager | 155 500 | 155 500 | 0 |

**Le contrôle est parfait** : les pivots retrouvent exactement les médianes de l'analyse par métier. Cela confirme que `pivot_table()` ne perd aucune ligne et que les agrégations sont cohérentes.

#### 8. Rappel méthodologique

Ces pivots montrent des **associations**, pas des effets causaux. L'effet « continent » est confondu avec :

- l'effet « niveau d'expérience » (composition : plus de seniors en Amérique du Nord) ;
- l'effet « métier » (structure des emplois : plus de SWE et Data Scientist aux US) ;
- l'effet « année » (94 % des lignes en 2024–2025, mais la distribution temporelle varie peut-être par continent) ;
- l'effet « taille d'entreprise » (les grandes entreprises américaines paient plus, et elles sont sur-représentées).

La modélisation (phase 6) isolera les effets propres de chaque variable, mais **ne prouvera jamais la causalité** sans design expérimental. Un salaire plus élevé aux US ne signifie pas que « vivre aux US cause un salaire plus élevé » : cela peut refléter un marché du travail plus tendu, un coût de la vie plus élevé, une sélection des profils, etc.

#### 9. Décisions pour la suite

- **Les pivots `experience_level × residence_continent` et `job_title × experience_level`** seront réutilisés directement pour les **heatmaps** de la phase 5 (étape 28). `sns.heatmap(pivot_median)` produit une visualisation immédiatement lisible.
- **Le pivot `job_title × experience_level`** servira à **valider le regroupement `job_category`** à l'étape 32 : les 10 métiers dominants ont des profils de progression très différents (Data Engineer 1,89× vs Research Scientist 1,10×). Un regroupement trop grossier (par exemple « tech » vs « business ») perdrait cette information. Il faudra probablement distinguer :
  - les métiers techniques à forte progression (Data Engineer, Engineer, SWE) ;
  - les métiers techniques à entrée élevée (Research Scientist, ML Engineer) ;
  - les métiers d'analyse (Data Analyst, Analyst) ;
  - les métiers de management (Manager, Product Manager).
- **La sur-représentation US** (93,7 % en Amérique du Nord) et **la structure plus senior de l'Amérique du Nord** (58,9 % de SE) seront mentionnées dans les limites du README final (étape 59). Le modèle risque de prédire essentiellement des salaires américains ; sa généralisation aux autres marchés est incertaine.
- **Les cellules à effectif < 30** seront explicitement exclues de toute analyse comparative dans les phases suivantes. Une fonction utilitaire pour filtrer les effectifs faibles pourra être ajoutée dans `src/salary_prediction/` à la phase 7.

## 8. Export des données traitées

### 8.1 Objectif

Exporter `df_clean` (13 colonnes, 151 445 lignes) vers
`data/processed/salaries_clean.csv`, puis vérifier que le fichier se recharge
à l'identique.

**Ce que contient le fichier exporté :**

- les 11 colonnes d'origine (audit possible) ;
- les 2 colonnes ajoutées : `residence_continent`, `company_continent` ;
- **les 79 532 doublons complets sont conservés** (déduplication au moment du
  split, étape 33) ;
- **aucune valeur extrême supprimée** (toutes les valeurs sont légitimes) ;
- **aucune valeur manquante** (0 depuis le chargement).

**Ce que ne contient PAS le fichier :**

- les colonnes `job_category` et `salary_in_usd_log` (créées à l'étape 32,
  après le split train/test) ;
- les colonnes issues du split train/test (générées à l'étape 33).

**Versionnement :** ce fichier est exclu par `.gitignore` (règle
`data/processed/*.csv`). Il est régénérable en réexécutant ce notebook.
On versionne le code, pas les données dérivées.

In [31]:
# Cellule 8.2 : export + contrôle de roundtrip
from pathlib import Path

# Chemin de sortie
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = PROCESSED_DIR / "salaries_clean.csv"

# Export (UTF-8, index=False pour ne pas polluer le CSV avec l'index Pandas)
df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

# Vérification de l'existence et de la taille
assert OUTPUT_PATH.exists(), "Le fichier exporté n'existe pas"
taille_mo = OUTPUT_PATH.stat().st_size / 1024**2
print(f"✔ Fichier exporté : {OUTPUT_PATH}")
print(f"  Taille : {taille_mo:.2f} Mo")

# Contrôle de roundtrip : on recharge et on compare
df_reload = pd.read_csv(OUTPUT_PATH, encoding="utf-8")
print(f"  Shape après rechargement : {df_reload.shape}")
assert df_reload.shape == (151445, 13), f"Shape inattendue : {df_reload.shape}"

# Comparaison des colonnes
assert list(df_reload.columns) == list(df_clean.columns), "Colonnes différentes"
print(f"  Colonnes identiques : ✔")

# Comparaison ligne à ligne sur les colonnes numériques
for col in ["work_year", "salary", "salary_in_usd", "remote_ratio"]:
    assert (df_reload[col] == df_clean[col]).all(), f"Différence sur {col}"
print(f"  Colonnes numériques identiques : ✔")

# Comparaison des colonnes catégorielles (les NaN éventuels sont gérés par isna)
for col in ["experience_level", "employment_type", "job_title",
            "salary_currency", "employee_residence", "company_location",
            "company_size", "residence_continent", "company_continent"]:
    assert (df_reload[col] == df_clean[col]).all(), f"Différence sur {col}"
print(f"  Colonnes catégorielles identiques : ✔")

print()
print("=== Résumé de l'export ===")
print(f"Lignes exportées       : {len(df_clean)}")
print(f"Colonnes exportées     : {len(df_clean.columns)}")
print(f"Valeurs manquantes     : {int(df_clean.isna().sum().sum())}")
print(f"Doublons conservés     : {int(df_clean.duplicated().sum())}")
print(f"Taille fichier         : {taille_mo:.2f} Mo")

✔ Fichier exporté : c:\Users\malak\Desktop\data-science-salary-analysis\data\processed\salaries_clean.csv
  Taille : 11.96 Mo
  Shape après rechargement : (151445, 13)
  Colonnes identiques : ✔
  Colonnes numériques identiques : ✔
  Colonnes catégorielles identiques : ✔

=== Résumé de l'export ===
Lignes exportées       : 151445
Colonnes exportées     : 13
Valeurs manquantes     : 0
Doublons conservés     : 79532
Taille fichier         : 11.96 Mo


In [32]:
# Cellule 8.3 : contrôle final — les stats clés sont inchangées après roundtrip
print("=== Contrôle : statistiques de salary_in_usd avant et après rechargement ===")

stats_avant = df_clean["salary_in_usd"].agg(["count", "mean", "median", "min", "max"])
stats_apres = df_reload["salary_in_usd"].agg(["count", "mean", "median", "min", "max"])

comparison = pd.DataFrame({
    "avant export": stats_avant,
    "après rechargement": stats_apres,
    "différence": (stats_avant - stats_apres).abs(),
})
print(comparison.round(4))
print()
assert (comparison["différence"] < 0.01).all(), "Statistiques différentes après rechargement !"
print("✔ Toutes les statistiques sont identiques à 0,01 près")

# Répartition des continents après rechargement
print()
print("=== Répartition par continent (après rechargement) ===")
print(df_reload["residence_continent"].value_counts())

=== Contrôle : statistiques de salary_in_usd avant et après rechargement ===
        avant export  après rechargement  différence
count    151445.0000         151445.0000         0.0
mean     157527.4584         157527.4584         0.0
median   146100.0000         146100.0000         0.0
min       15000.0000          15000.0000         0.0
max      800000.0000         800000.0000         0.0

✔ Toutes les statistiques sont identiques à 0,01 près

=== Répartition par continent (après rechargement) ===
residence_continent
North America    141839
Europe             7864
Oceania             671
Asia                491
South America       335
Africa              169
Middle East          76
Name: count, dtype: int64
